In [ ]:
import os
os.chdir('/content/')
print(os.getcwd())  # confirm

In [ ]:
# Installing some pre-requirements
! apt-get install gnuplot

In [ ]:
! pip install spirit

In [ ]:
! pip install gr

In [ ]:
!pip install pymupdf

In [ ]:
# Clone the SWIS spin-wave package @flavianojs github
! git clone https://github.com/flavianojs/SWIS.git

In [ ]:
! cp -r SWIS/mininum_folder_setup my_spinwaves_study

In [ ]:
import os
os.chdir('my_spinwaves_study')
print(os.getcwd())  # confirm

Rename 'input_XXXX.cfg' and to 'input_FM.cfg', and similarly to 'inputcard_XXXX.inp'

In [ ]:
! mv input_XXXX.cfg input_FM.cfg
! mv inputcard_XXXX.inp inputcard_FM.inp

The code has three main input files:
1. runspinwave.sh is the master bash script that controls what is goint to be run
2. inputcard_*.inp is the SWIS input card
3. input_*.cfg is the Spirit (spin dynamics code) input card

Modify my_spinwave_study/runspinwave.sh to update the source_folder variable to source_folder=/content/SWIS

Update the variable at runspinwave.sh spirit_config_file='input_FM.cfg'

Run the first time to compile the code:

In [ ]:
! ./runspinwave.sh FM

Turn on the following options at runspinwave.sh:

```
    groundstate=0
        compile=1
      executing=1
 forceexecuting=0
        lattice=1
     occupation=0
          kpath=1
 dispersionplot=1
         spirit=1
     scale_pair=0
```




In [ ]:
! ./runspinwave.sh FM

SWIS plots the spin-wave spectrum on a file named disp_unfol*.png

In [ ]:
from IPython.display import Image, display
display(Image('disp_unfolpY_FM.png'))

The functions below help you to plot the spectrum in python:

In [ ]:
"""
Plot a spin-wave spectrum heat map (unfolded intensity + raw dispersion lines),
translating the logic of plot_dispersion.gnu into matplotlib.

Expected file formats (as in the original gnuplot script):

matriz1 ("disp_unfol_<specifier>.dat")
    Columns (1-indexed, whitespace separated):
      1: q-path coordinate (x)
      2: energy / frequency, omega in meV (y)
      7,8,9,10: spectral-weight components for one polarization; the
                gnuplot script colors the heat map by their sum.
    Organized as a gnuplot pm3d grid: blocks of constant x separated by
    blank lines, with y varying within each block.

fort1 ("dispersion_<specifier>.dat")
    Column 1: q-path coordinate (x)
    Columns 2..N: raw (folded) magnon band energies, one column per mode.
    Plotted as thin lines on top of the heat map.

highsympoints.gnu
    Optional. Not parsed here (it's gnuplot syntax); pass the high-symmetry
    tick positions/labels explicitly instead (see `xtick_positions`,
    `xtick_labels` below). Defaults mirror the values hardcoded in the
    original script (L, Gamma, X, W, Gamma).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_spin_wave_spectrum(
    matriz1_file,
    dispersion_file=None,
    ymin=None,
    ymax=None,
    cbmax=None,
    intensity_cols=(6, 7, 8, 9),  # 0-indexed -> gnuplot columns 7,8,9,10
    xtick_positions=None,
    xtick_labels=None,
    cmap="gnuplot",
    figsize=(9, 5),
    overlay_lines=True,
    line_color="lime",
    line_width=0.4,
    output=None,
    dpi=150,
):
    """
    Plot a heat map of the (unfolded) spin-wave spectral function, with the
    raw dispersion bands overlaid as thin lines -- equivalent to the
    `splot ... w pm3d` + `w lines` combination in plot_dispersion.gnu.

    Parameters
    ----------
    matriz1_file : str
        Path to the unfolded spectral-weight file (gnuplot pm3d grid format:
        blank-line-separated blocks of constant x).
    dispersion_file : str or None
        Path to the raw dispersion file (fort1 in the gnuplot script).
        If None, no overlay lines are drawn.
    ymin, ymax : float or None
        Energy axis limits (meV). Defaults to the data range if None.
    cbmax : float or None
        Colorbar max (mirrors `cbmax` in the script). If None, uses the
        data max.
    intensity_cols : tuple of int
        0-indexed columns of matriz1_file to sum for the heat-map intensity
        (default matches gnuplot's $7+$8+$9+$10).
    xtick_positions, xtick_labels : sequence
        High-symmetry point positions along the q-path and their labels
        (defaults reproduce the L-Gamma-X-W-Gamma path hardcoded in the
        original script; pass your own if your path differs).
    cmap : str
        Matplotlib colormap (gnuplot's default pm3d palette is closer to
        'jet'/'turbo'; 'inferno' is a more perceptually-uniform substitute).
    overlay_lines : bool
        Whether to draw the raw dispersion bands from dispersion_file.
    output : str or None
        If given, save the figure to this path (e.g. "dispersion.png").

    Returns
    -------
    fig, ax : matplotlib Figure and Axes
    """
    # --- load and grid the unfolded spectral weight -----------------------
    data = pd.read_csv(
        matriz1_file, sep=r"\s+", comment="#", header=None
    )
    x = data[0].to_numpy()
    y = data[1].to_numpy()
    intensity = data[list(intensity_cols)].sum(axis=1).to_numpy()

    # Reshape the (possibly block-wise, gnuplot pm3d style) scattered data
    # onto a regular (x, y) grid.
    x_unique, x_idx = np.unique(x, return_inverse=True)
    y_unique, y_idx = np.unique(y, return_inverse=True)
    Z = np.full((len(y_unique), len(x_unique)), np.nan)
    Z[y_idx, x_idx] = intensity

    if cbmax is None:
        cbmax = np.nanmax(Z)

    # --- plot ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)
    mesh = ax.pcolormesh(
        x_unique, y_unique, Z, shading="gouraud", cmap=cmap, vmin=0, vmax=cbmax
    )
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_ticks([0, cbmax])
    cbar.set_ticklabels(["0", "max"])

    # --- overlay raw dispersion bands -----------------------------------
    if overlay_lines and dispersion_file is not None:
        disp = pd.read_csv(
            dispersion_file, sep=r"\s+", comment="#", header=None
        )
        qpath = disp[0].to_numpy()
        for col in disp.columns[1:]:
            ax.plot(
                qpath,
                disp[col].to_numpy(),
                color=line_color,
                linewidth=line_width,
            )

    # --- axes cosmetics --------------------------------------------------
    ax.set_ylabel(r"$\omega$ (meV)")
    ax.set_ylim(
        ymin if ymin is not None else y_unique.min(),
        ymax if ymax is not None else y_unique.max(),
    )
    if xtick_positions is not None:
        ax.set_xticks(xtick_positions)
        ax.set_xticklabels(xtick_labels)
        for pos in xtick_positions:
            ax.axvline(pos, color="white", linewidth=0.4, linestyle="-")
    ax.set_xlim(x_unique.min(), x_unique.max())

    fig.tight_layout()
    if output is not None:
        fig.savefig(output, dpi=dpi)

    return fig, ax


In [ ]:
TAG='FM'
fig, ax = plot_spin_wave_spectrum(
    matriz1_file="outputfiles/disp_unfol_"+TAG+".dat",
    dispersion_file="outputfiles/dispersion_"+TAG+".dat",
    ymin=None,
    ymax=None,
    cbmax=None,
    output="disp_unfolpY_"+TAG+"_python.png",
)
plt.show()

List of tasks:
1. Run a 1-dimension ferromagnetic spin chain
2. What happens to the spin-wave dispersion if we apply an external magnetic field? Remember to modify the SWIS inputcard and Spirit's inputcard.
3. How about anisotropy?
4. Now try to simulate an antiferromagnet. You will need to modify the exchange parameter files: inputfiles/Jij_pair.spirit and to expand the unit cell on the Spirit inputfile input_XX.cfg, variable n_basis_cells.
5. Try a 2-dimension system. Set the kpoint path throughout the 2D Brillouin-zone.
6. Time to try to add the Dzyaloshinskii-Moriya interaction (DMI) to form spin spirals.
7. Can you try to form a skyrmion lattice?


In [ ]:
! cp inputcard_FM.f90 inputcard_FM.inp
! ./runspinwave.sh FM

In [ ]:

TAG='FM'
fig, ax = plot_spin_wave_spectrum(
    matriz1_file="outputfiles/disp_unfol_"+TAG+".dat",
    dispersion_file="outputfiles/dispersion_"+TAG+".dat",
    ymin=None,
    ymax=None,
    cbmax=None,
    output="disp_unfolpY_"+TAG+"_python.png",
)
plt.show()

It also creates a plot of the kpoint path

In [ ]:

import fitz
doc = fitz.open('kpath_FM.pdf')
for page in doc:
    pix = page.get_pixmap(dpi=200)
    display(Image(data=pix.tobytes('png')))

We can also visualize the ground state spin configuration

In [ ]:
display(Image('spinconfig_final.png'))